Inspect the data inside the truthtriplet files.

These are the upstream data files we will make the prong embed training data from.

To make the file that goes into this notebook run `run_lardata2hdf5.py`

Script found in `ubdl/larflow/larmatchnet/larmatch/`
```
python3 run_lardata2hdf5.py --input-larlite [larlite_file.root] --input-larcv [larcv_file.root] -o [outfile.h5] -tb -tri
```

In [ ]:
import chart_studio as cs
import chart_studio.plotly as py
import plotly.graph_objects as go
from ctypes import c_int
import numpy as np
from larmatch.data.larmatch_hdf5_reader import LArMatchHDF5Dataset, get_data_loader
%load_ext autoreload
%autoreload 2

In [ ]:
# data useful for plots
import lardly
from lardly import DetectorOutline # load utility to draw TPC outline
detdata = DetectorOutline()
detlines = detdata.getlines(color=(10,10,10))

# PARTICLE LABEL COLORS
ssnetcolor = {0:np.array((0,0,0)),     # ghost                                                                                                                                                   
              1:np.array((255,0,0)),   # electron                                                                                                                                       
              2:np.array((0,255,0)),   # gamma                                                                                                                             
              3:np.array((0,0,255)),   # muon                                                                                                                                              
              4:np.array((255,0,255)), # proton                                                                                                                                                 
              5:np.array((0,255,255)) # pion (+other mesons)
             }


ssnetnames = {0:"ghost",
             1:"e",
             2:"gamma",
             3:"mu",
             4:"proton",
             5:"pion"}

kpcolors = {0:np.array((255,0,0)), # nu (red)
            1:np.array((0,255,0)), # track-start (green)
            2:np.array((0,0,255)), # track-end (blue)
            3:np.array((255,128,0)), # shower (orange)
            4:np.array((0,255,255)), # michel
            5:np.array((255,0,255))} # delta
kpnames = {0:"neutrino",
          1:"track-start",
          2:"track-end",
          3:"shower-start",
          4:"michel-start",
          5:"delta-start"}


In [ ]:
inputfiles = ["test.h5"]

In [ ]:
dataset = get_data_loader( inputfiles, batch_size=1, shuffle=False, num_workers=1)
iter_data = iter(dataset)

In [ ]:
data = next(iter_data)

In [ ]:
batch_size = len(data)
print(data[0].keys())

ibatch = 0
batchdata = data[ibatch]

In [ ]:
# extract the true keypoint locations
kppos  = batchdata['keypoint_truth_pos']
kptype = batchdata['keypoint_truth_kptype_pdg_trackid'][:,0]
kppdg  = batchdata['keypoint_truth_kptype_pdg_trackid'][:,1]
kptid  = batchdata['keypoint_truth_kptype_pdg_trackid'][:,2]
keypoint_plots = []
for ikptype in kpnames:
    mask = kptype==ikptype
    pos = kppos[ mask ]
    xpdg = kppdg[mask]
    xtid = kptid[mask]
    print("Number of %s keypoints: %d"%(kpnames[ikptype],pos.shape[0]))
    hovertemplate='<b>PDG</b>: %{customdata[0]}<br>' + \
                  '<b>Tid</b>: %{customdata[1]}<br>' + \
                  'x: %{x}<br>' + \
                  'y: %{y}<br>' + \
                  'z: %{z}<br>'
    kptype_plot = {
    "type":"scatter3d",
    "x": pos[:,0],
    "y": pos[:,1],
    "z": pos[:,2],
    "mode":"markers",
    "name":kpnames[ikptype],
    "customdata":np.stack( (xpdg,xtid), axis=-1 ),
    "marker":{"color":"rgb(%d,%d,%d)"%tuple(kpcolors[ikptype]),"size":5.0,"opacity":0.5},
    "hovertemplate":hovertemplate
    }
    if kpnames[ikptype]=="neutrino":
        kptype_plot['marker']['size'] = 10.0
    keypoint_plots.append( kptype_plot )
    

In [ ]:
# Plot the true/ghost labels

# use a limit to speed up plotting (will randomly sample)
LIMIT_NUM_SPACEPOINTS=True
MAX_NUM_SAMPLES=10000

# get the arrays
triplets   = batchdata['matchtriplet']
spacepoints = batchdata['spacepoints']
print("spacepoints.shape: ",spacepoints.shape)
npts = spacepoints.shape[0]
truthlabel  = triplets[:npts,3]
print("triplets.shape: ",triplets.shape)

if LIMIT_NUM_SPACEPOINTS:
    xrandom = np.random.random(npts)<0.5
    xpos = spacepoints[xrandom[:],:]
    xtruth = truthlabel[xrandom[:]]
else:
    xpos = spacepoints
    xtruth = truthlabel
#print(spacepoints[:10])
#print(truthlabel[:10])

# define a 3d scatter plot in plotly
plot_spacepoints = {
    "type":"scatter3d",
    "x": xpos[:,0],
    "y": xpos[:,1],
    "z": xpos[:,2],
    "mode":"markers",
    "name":"spacepoints",
    "marker":{"color":xtruth.astype(np.float64),"size":1.0,"opacity":0.5,"colorscale":"Bluered"},
    }


# collect the things we want to plot
# we add the spacepoint plot to the outline of the uboone detector
plot_list = detlines + [plot_spacepoints]
try:
    plot_list += keypoint_plots
except:
    print("no keypoint_plots?")

# define axes and layout of plot
axis_template = {
    "showbackground": True,
    "backgroundcolor": "rgba(100, 100, 100,0.5)",
    "gridcolor": "rgb(50, 50, 50)",
    "zerolinecolor": "rgb(0, 0, 0)",
}
layout = go.Layout(
    title='Truth/Ghost Labels',
    autosize=True,
    hovermode='closest',
    showlegend=False,
    scene= {
        "xaxis": axis_template,
        "yaxis": axis_template,
        "zaxis": axis_template,
        "aspectratio": {"x": 1, "y": 1, "z": 3},
        "camera": {"eye": {"x": -2, "y": 0.25, "z": 0.0},
                   "center":dict(x=0, y=0, z=0),
                   "up":dict(x=0, y=1, z=0)},
        "annotations": [],
    }
)


# make the plot using plotly go
fig = go.Figure(data=plot_list, layout=layout)
fig.show()

In [ ]:
# PLOT SSNET LABELS

ssnetdata = batchdata['ssnet_label']
spacepoints = batchdata['spacepoints']
print("ssnetdata.shape: ",ssnetdata.shape)

SSNET_LIMIT_NUM_SPACEPOINTS=True
SSNET_MAX_NUM_SAMPLES=10000

ssnet_plots = []

# first isolate by class
for iclass in ssnetcolor:
    xmask = ssnetdata==iclass
    xlabels = ssnetdata[xmask]
    xpos = spacepoints[xmask[:],:]
    # downsample if needed
    if SSNET_LIMIT_NUM_SPACEPOINTS and xlabels.shape[0]>SSNET_MAX_NUM_SAMPLES:
        factor = float(SSNET_MAX_NUM_SAMPLES)/float(xlabels.shape[0])
        xfilter = np.random.random( xlabels.shape[0] ) < factor
        xlabels = xlabels[ xfilter ]
        xpos = xpos[ xfilter[:], :]
    
    print("number of ssnet class [",ssnetnames[iclass],"] spacepoints: ",xpos.shape[0])
    if xpos.shape[0]==0:
        continue
    ptsize = 1.0
    if iclass==0:
        # reduce ghost point size
        ptsize = 0.5
        
    ssnet_plot = {
    "type":"scatter3d",
    "x": xpos[:,0],
    "y": xpos[:,1],
    "z": xpos[:,2],
    "mode":"markers",
    "name":"[%d] %s"%(iclass,ssnetnames[iclass]),
    "marker":{"color":"rgb(%d,%d,%d)"%tuple(ssnetcolor[iclass]),"size":ptsize,"opacity":0.5},
    }
    ssnet_plots.append( ssnet_plot )

ssnet_plot_traces = detlines + ssnet_plots
try:
    ssnet_plot_traces += keypoint_plots
except:
    print("no keypoint plots")

layout = go.Layout(
    title='SSNet Labeled Spacepoints',
    autosize=True,
    hovermode='closest',
    showlegend=False,
    scene= {
        "xaxis": axis_template,
        "yaxis": axis_template,
        "zaxis": axis_template,
        "aspectratio": {"x": 1, "y": 1, "z": 3},
        "camera": {"eye": {"x": -2, "y": 0.25, "z": 0.0},
                   "center":dict(x=0, y=0, z=0),
                   "up":dict(x=0, y=1, z=0)},
        "annotations": [],
    }
)

fig = go.Figure(data=ssnet_plot_traces, layout=layout)
fig.show()


In [ ]:
# Plot by origin label

origin = batchdata['origin_label']
spacepoints = batchdata['spacepoints']
print(np.unique(origin))
print("origin.shape: ",origin.shape)

origin_names = {0:'ghost',
               1:'neutrino',
               2:'cosmic'}
origin_colors = {0:'rgb(0,0,0)',
                1:'rgb(255,0,0)',
                2:'rgb(0,0,255)'}

ORIGIN_LIMIT_NUM_SPACEPOINTS=True
ORIGIN_MAX_NUM_SAMPLES=10000

origin_plots = []

for iorigin in [0,1,2]:
    xmask = origin==iorigin
    xpos = spacepoints[xmask[:],:]
    if ORIGIN_LIMIT_NUM_SPACEPOINTS and xpos.shape[0]>ORIGIN_MAX_NUM_SAMPLES:
        factor = float(ORIGIN_MAX_NUM_SAMPLES)/float(xpos.shape[0])
        xfilter = np.random.random( xpos.shape[0] ) < factor
        xpos = xpos[xfilter[:],:]
        
    ptsize = 1.0
    if iorigin==0:
        ptsize = 0.5
    plot = {
    "type":"scatter3d",
    "x": xpos[:,0],
    "y": xpos[:,1],
    "z": xpos[:,2],
    "mode":"markers",
    "name":"[%d] %s"%(iorigin,origin_names[iorigin]),
    "marker":{"color":origin_colors[iorigin],"size":ptsize,"opacity":0.5},
    }
    origin_plots.append( plot )

origin_plot_traces = detlines + origin_plots
try:
    origin_plot_traces += keypoint_plots
except:
    print("no keypoint plots")

layout = go.Layout(
    title='Origin labeled Spacepoints',
    autosize=True,
    hovermode='closest',
    showlegend=False,
    scene= {
        "xaxis": axis_template,
        "yaxis": axis_template,
        "zaxis": axis_template,
        "aspectratio": {"x": 1, "y": 1, "z": 3},
        "camera": {"eye": {"x": -2, "y": 0.25, "z": 0.0},
                   "center":dict(x=0, y=0, z=0),
                   "up":dict(x=0, y=1, z=0)},
        "annotations": [],
    }
)

fig = go.Figure(data=origin_plot_traces, layout=layout)
fig.show()


In [ ]:
# Direction labels

pf = batchdata['paf_label']
spacepoints = batchdata['spacepoints']
truemask = batchdata['matchtriplet'][:,3]==1
pftrue = pf[truemask[:],:]
sptrue = spacepoints[truemask[:],:]

print('pftrue.shape: ',pftrue.shape)

PF_LIMIT_NUM_SPACEPOINTS=True
PF_MAX_NUM_SAMPLES=10000

if PF_LIMIT_NUM_SPACEPOINTS and pftrue.shape[0]>PF_MAX_NUM_SAMPLES:
    factor = float(PF_MAX_NUM_SAMPLES)/float(pftrue.shape[0])
    xfilter = np.random.random( pftrue.shape[0] ) < factor
    xpftrue = pftrue[xfilter[:],:]
    xsptrue = sptrue[xfilter[:],:]
else:
    xpftrue = pftrue
    xsptrue = sptrue

#print(xpftrue[:10,])
print(xsptrue[:10,])

print("num paf spacepoints: ",xpftrue.shape)

pf_plot = {
    "type":"cone",
    "x": xsptrue[:,0],
    "y": xsptrue[:,1],
    "z": xsptrue[:,2],
    "u": xpftrue[:,0],
    "v": xpftrue[:,1],
    "w": xpftrue[:,2],
    "name":"pflow",
    "sizemode":"absolute",
    "sizeref":15,
    "anchor":"tail"
    }

pf_traces = detlines + [pf_plot]

layout = go.Layout(
    title='Particle Flow Direction',
    autosize=True,
    hovermode='closest',
    showlegend=False,
    scene= {
        "xaxis": axis_template,
        "yaxis": axis_template,
        "zaxis": axis_template,
        "aspectratio": {"x": 1, "y": 1, "z": 3},
        "camera": {"eye": {"x": -2, "y": 0.25, "z": 0.0},
                   "center":dict(x=0, y=0, z=0),
                   "up":dict(x=0, y=1, z=0)},
        "annotations": [],
    }
)

fig = go.Figure(data=pf_traces, layout=layout)
fig.show()